In [ ]:
!pip install -q \

langchain \

langchain-core \

langchain-community \

langchain-google-genai \

faiss-cpu>=1.9.0 \

sentence-transformers \

langchain-classic \

langchain-text-splitters \

langchain-huggingface

In [1]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import  FAISS
from langchain_core.documents import Document
from dotenv import load_dotenv

C:\Users\vaish\AppData\Local\Temp\ipykernel_23712\2406591703.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import  FAISS


In [2]:
load_dotenv()
api_key=os.getenv("GEMINI_API_KEY")


In [3]:
llm =ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",temperature=0.0
)

embedder=GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [4]:
audit_data = [

Document(page_content="MLOps Audit Q4: European division legacy branches report a 15% OCR failure rate.", metadata={"id": 1}),
Document(page_content="Tuesday Review confirmed the 15% spike is due to 'Legacy Scan-X' hardware and firmware v2.1.", metadata={"id": 2}),
Document(page_content="Jaymin approved a $45,000 emergency budget to upgrade European scanners by Q1 end.", metadata={"id": 3}),
Document(page_content="OCR failures peak on Tuesdays due to weekly bulk-batch processing of handwritten PDFs.", metadata={"id": 4}),
Document(page_content="Tony recommends a distributed architecture for handling 500+ PDFs in legacy branches.", metadata={"id": 5}),
Document(page_content="The 15% error rate is classified as 'Critical' for Banking and Compliance audits.", metadata={"id": 6}),
Document(page_content="Marten's team is monitoring OCR logs 24/7 until the hardware upgrade is finished.", metadata={"id": 7}),
Document(page_content="European legacy branches are the only units still using the v2.1 firmware.", metadata={"id": 8}),
Document(page_content="The new firmware v3.0 has been successfully tested in the North American cluster.", metadata={"id": 9}),
Document(page_content="Budget allocation for Q1 also includes a 10% reserve for unexpected cloud egress costs.", metadata={"id": 10}),
Document(page_content="Anisha suggested moving OCR processing to an asynchronous queue using RabbitMQ.", metadata={"id": 11}),
Document(page_content="Legacy Scan-X machines have a known overheating issue when processing over 100 pages.", metadata={"id": 12}),
Document(page_content="Compliance team noted that OCR errors are leading to incorrect data in customer KYC files.", metadata={"id": 13}),
Document(page_content="The upgrade project is codenamed 'Project Vision' and is led by the MLOps Core team.", metadata={"id": 14}),
Document(page_content="Handwritten PDF recognition accuracy dropped to 62% in the last batch test.", metadata={"id": 15}),
Document(page_content="Security audit found that legacy firmware v2.1 has three unpatched vulnerabilities.", metadata={"id": 16}),
Document(page_content="Training data for the new OCR model includes 50,000 samples of handwritten European scripts.", metadata={"id": 17}),
Document(page_content="The hardware vendor 'OptiScan' has been notified about the hardware failures.", metadata={"id": 18}),
Document(page_content="A temporary patch was deployed on Monday to reduce memory leaks during batch processing.", metadata={"id": 19}),
Document(page_content="Q2 Roadmap: Complete migration of all legacy branches to the centralized MLOps platform.", metadata={"id": 20})

]

In [ ]:
vectorstore=FAISS.from_documents(audit_data,embedder)
base_retriever=vectorstore.as_retriever(search_kwargs={"k":15})#bi encoder

In [6]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder


# 1 .Initialize the model

cross_encoder_model=HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")

#2.Configure the re ranker

reranker= CrossEncoderReranker(model=cross_encoder_model,top_n=3)

#3.BUILD 2nd stage retriever
compression_retriever=ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever
)

c:\Users\vaish\agenticaisep\.venv13\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4716.84it/s]


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough  #Take the input you received and pass it through unchanged.

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)



rag_prompt=ChatPromptTemplate.from_template(
    """You are an elite MLOPs auding algorithm.Answer using ONLY this highly-vetted context:
    {context}
    Question:{question}
    Answer:"""
)

In [8]:
#Advanced RAG Chain

advanced_rerank_chain=(
{"context":compression_retriever | format_docs,"question":RunnablePassthrough()}
|rag_prompt
|llm
|StrOutputParser()
)

In [12]:
user_query = " What is the exact OCR failure rate mentioned in Q4 audit?"

final_answer = advanced_rerank_chain.invoke(user_query)

print("User Query:", user_query)

print("="*40)

print("Final Answer:", final_answer)

print(compression_retriever)

c:\Users\vaish\agenticaisep\.venv13\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


User Query:  What is the exact OCR failure rate mentioned in Q4 audit?
Final Answer: 15%
base_compressor=CrossEncoderReranker(model=HuggingFaceCrossEncoder(client=CrossEncoder(
  (0): Transformer({'transformer_task': 'sequence-classification', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'logits'}}, 'module_output_name': 'scores', 'architecture': 'BertForSequenceClassification'})
), model_name='cross-encoder/ms-marco-MiniLM-L-6-v2', model_kwargs={}), top_n=3) base_retriever=VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001EFFEF36660>, search_kwargs={'k': 15})


In [ ]:
#fromat_docs--hae contents of compresiion retriever,takes compression_retriever as docs ,compression_rerteiever has metadta,pagecontent 

In [17]:
query = "What is the exact OCR failure rate mentioned in Q4 audit?"

docs = compression_retriever.invoke(query)

for i, doc in enumerate(docs, 1):
    print(f"\n--- Document {i} ---")
    print(doc.page_content,doc.metadata)


--- Document 1 ---
MLOps Audit Q4: European division legacy branches report a 15% OCR failure rate. {'id': 1}

--- Document 2 ---
The 15% error rate is classified as 'Critical' for Banking and Compliance audits. {'id': 6}

--- Document 3 ---
OCR failures peak on Tuesdays due to weekly bulk-batch processing of handwritten PDFs. {'id': 4}


In [18]:
context = format_docs(docs)

print("\n===== FINAL CONTEXT SENT TO LLM =====")
print(context)


===== FINAL CONTEXT SENT TO LLM =====
MLOps Audit Q4: European division legacy branches report a 15% OCR failure rate.

The 15% error rate is classified as 'Critical' for Banking and Compliance audits.

OCR failures peak on Tuesdays due to weekly bulk-batch processing of handwritten PDFs.
